In [3]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# -------------------------
# 1. Load datasets
# -------------------------
sales = pd.read_csv("Sales.csv")
customers = pd.read_csv("Customers.csv")
models = pd.read_csv("MobileModels.csv")

sales['sale_date'] = pd.to_datetime(sales['sale_date'])

# -------------------------
# 2. KPI CALCULATIONS
# -------------------------
total_revenue  = sales['final_price'].sum()
total_customers = customers['customer_id'].nunique()
total_sales = sales['sale_id'].nunique()

# -------------------------
# 3. DATA FOR CHARTS
# -------------------------
brand_sales = sales.merge(models[['model_id','brand']], on='model_id') \
                   .groupby("brand")["final_price"].sum().reset_index()

city_sales = sales.merge(customers[['customer_id','city']], on='customer_id') \
                  .groupby("city")["final_price"].sum().reset_index()

monthly = sales.groupby(sales["sale_date"].dt.to_period("M"))["final_price"].sum()
monthly.index = monthly.index.to_timestamp()

# -------------------------
# 4. CREATE DASHBOARD
# -------------------------
fig = make_subplots(
    rows=4, cols=2,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"colspan": 2, "type": "scatter"}, None]
    ],
    subplot_titles=[
        "Total Revenue", "Total Customers",
        "Revenue by Brand", "Revenue by City",
        "Discount vs Final Price", "Average Sale Value",
        "Monthly Revenue Trend"
    ],
    horizontal_spacing=0.1,
    vertical_spacing=0.12
)

# -------------------------
# 5. KPI CARDS
# -------------------------
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=total_revenue,
    number={'prefix': "₹", 'font': {'size': 28}},
    delta={'reference': total_revenue*0.8, 'relative': True, 'position': "top"},
    title={"text": "Revenue"}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_customers,
    number={'font': {'size': 28}},
    title={"text": "Customers"}
), row=1, col=2)

# -------------------------
# 6. BAR CHARTS
# -------------------------
fig.add_trace(go.Bar(
    x=brand_sales['brand'], y=brand_sales['final_price'],
    marker_color="#0083B8", text=brand_sales['final_price'], textposition='auto'
), row=2, col=1)

fig.add_trace(go.Bar(
    x=city_sales['city'], y=city_sales['final_price'],
    marker_color="#EF553B", text=city_sales['final_price'], textposition='auto'
), row=2, col=2)

# -------------------------
# 7. SCATTER PLOTS
# -------------------------
fig.add_trace(go.Scatter(
    x=sales["discount"], y=sales["final_price"],
    mode="markers",
    marker=dict(size=7, color="#AB63FA"),
    hovertemplate="Discount: %{x}<br>Price: ₹%{y}"
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=sales["final_price"], y=sales["final_price"]/sales["final_price"].mean(),
    mode="markers",
    marker=dict(size=7, color="#FFA15A"),
    hovertemplate="Sale: ₹%{x}<br>Ratio: %{y:.2f}"
), row=3, col=2)

# -------------------------
# 8. MONTHLY REVENUE TREND
# -------------------------
fig.add_trace(go.Scatter(
    x=monthly.index, y=monthly.values,
    mode="lines+markers",
    line=dict(color="#00CC96", width=3),
    hovertemplate="Month: %{x|%b %Y}<br>Revenue: ₹%{y}"
), row=4, col=1)

# -------------------------
# 9. LAYOUT SETTINGS
# -------------------------
fig.update_layout(
    title="📊 MOBILE SHOPPEE — PROFESSIONAL DASHBOARD",
    height=950,
    width=1400,  # wider for horizontal scroll
    template="plotly_white",
    showlegend=False,
    margin=dict(l=50, r=50, t=100, b=50),
    paper_bgcolor="#F5F5F5",
    plot_bgcolor="#FFFFFF",
    hovermode="closest",
    font=dict(family="Arial", size=12),
    shapes=[dict(
        type="rect",
        x0=0, y0=0, x1=1, y1=1,
        xref='paper', yref='paper',
        line=dict(color="black", width=2),
        layer="below"
    )]
)

# -------------------------
# 10. ENABLE ZOOM AND PAN
# -------------------------
fig.update_xaxes(fixedrange=False)
fig.update_yaxes(fixedrange=False)

# -------------------------
# 11. WRAP IN SCROLLABLE DIV
# -------------------------
scrollable_html = f"""
<div style="overflow:auto; border:2px solid black; width:100%; height:800px;">
    {fig.to_html(include_plotlyjs='cdn', full_html=False)}
</div>
"""

display(HTML(scrollable_html))
